# CareLoop Clinics — No-Show Guard

## Part 4 — Generative AI

**Objective:** Use the Gemini API to analyse call-centre notes, identify patient no-show reasons, validate responses using Pydantic, and generate recommendations.

**Tasks:**
- 4.1 One validated Gemini API call
- 4.2 Validate and process 10 notes
- 4.3 Test prompt weaknesses
- 4.4 Recommendations and GitHub

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
from google import genai
from google.genai import types

import json
import pandas as pd
import time

from pydantic import BaseModel, Field, ValidationError

In [ ]:
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [ ]:
MODEL_NAME = "gemini-flash-lite-latest"

SYSTEM_TEXT = """
You are a helpful assistant analysing patient appointment follow-up notes.
Identify the most likely reason for the missed appointment.
Use only the allowed root_cause values:
forgot_or_no_reminder, distance_or_transport, cost_or_fee,
illness_or_emergency, work_or_schedule, other.
Return only valid JSON matching the required schema.
Do not invent information.
"""

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}


def analyse_note(note_text):

    prompt = f"""
    Context: A clinic wants to understand why patients miss appointments.

    Role: You are a patient follow-up note analyst.

    Action: Analyse the following note and identify the most likely reason.

    Note: {note_text}

    Tone: Clear and professional.

    Format:
    Return JSON with root_cause, confidence, summary,
    and recommended_action.
    Use one of the allowed root_cause values.
    Keep summary within 25 words.
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
    system_instruction=SYSTEM_TEXT,
    temperature=0,
    max_output_tokens=1000,
    response_mime_type="application/json",
    response_schema=SCHEMA
)







            )



    return json.loads(response.text)

In [ ]:
import os

print([file for file in os.listdir() if "note" in file.lower()])

['follow up notes.csv']


In [ ]:
notes_df = pd.read_csv("follow up notes.csv")

print("Shape:", notes_df.shape)
display(notes_df.head(3))

Shape: (30, 3)


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...


In [ ]:
print(notes_df.columns.tolist())
display(notes_df.head(10))

['note_id', 'appointment_id', 'note_text']


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."
5,N006,A0507,Manager called an urgent meeting at the office...
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
7,N008,A0222,"City bus strike that day, no auto available ne..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."
9,N010,A0596,Had a shift change at the factory and could no...


In [ ]:
note_001 = notes_df.loc[
    notes_df["note_id"] == "N001", "note_text"
].iloc[0]

print(note_001)

Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.


In [ ]:
from typing import Literal

class NoteAnalysis(BaseModel):
    root_cause: Literal[
        "forgot_or_no_reminder",
        "distance_or_transport",
        "cost_or_fee",
        "illness_or_emergency",
        "work_or_schedule",
        "other"
    ]
    confidence: float = Field(ge=0, le=1)
    summary: str
    recommended_action: str

In [ ]:
raw_result = analyse_note(note_001)

validated_result = NoteAnalysis.model_validate(raw_result)

print(validated_result.model_dump())

{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.99, 'summary': 'Patient forgot the appointment because she did not receive any reminder message and wishes to reschedule.', 'recommended_action': "Reschedule the appointment for next week and verify the patient's contact information for future reminders."}


In [ ]:
test_response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Reply with only the word OK."
)

print(test_response.text)

OK


In [ ]:
print(validated_result.model_dump_json(indent=2))

{
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.99,
  "summary": "Patient forgot the appointment because she did not receive any reminder message and wishes to reschedule.",
  "recommended_action": "Reschedule the appointment for next week and verify the patient's contact information for future reminders."
}



## 4.2 — Batch Analysis of Call-Centre Notes

Process the first 10 call-centre notes using the Gemini API. Validate each response using Pydantic, skip blank notes, handle invalid responses, and preserve all records for further analysis.


In [ ]:
notes_to_process = notes_df.head(10).copy()

print("Notes to process:", len(notes_to_process))
print("Blank notes:", notes_to_process["note_text"].isna().sum())

Notes to process: 10
Blank notes: 1


In [ ]:
results = []

print("Results list created.")

Results list created.


In [ ]:
for row in notes_to_process.itertuples(index=False):

    if pd.isna(row.note_text) or not str(row.note_text).strip():
        results.append({
            "note_id": row.note_id,
            "appointment_id": row.appointment_id,
            "status": "skipped_blank"
        })
        continue

    try:
        raw_result = analyse_note(row.note_text)
        validated_result = NoteAnalysis.model_validate(raw_result)

        results.append({
            "note_id": row.note_id,
            "appointment_id": row.appointment_id,
            **validated_result.model_dump(),
            "status": "analysed"
        })

        print(row.note_id, "— analysed successfully")
        time.sleep(1)

    except Exception as e:
        results.append({
            "note_id": row.note_id,
            "appointment_id": row.appointment_id,
            "status": "invalid_response",
            "error": str(e)
        })

print("Total records:", len(results))

N001 — analysed successfully
N002 — analysed successfully
N005 — analysed successfully
N006 — analysed successfully
N007 — analysed successfully
N008 — analysed successfully
N009 — analysed successfully
Total records: 10


In [ ]:
print("Total records:", len(results))

print("\nStatus counts:")
print(pd.DataFrame(results)["status"].value_counts())

display(pd.DataFrame(results))

Total records: 10

Status counts:
status
analysed            7
invalid_response    2
skipped_blank       1
Name: count, dtype: int64


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,status,error
0,N001,A0872,forgot_or_no_reminder,0.95,Patient forgot the appointment because she did...,Reschedule the appointment for next week and v...,analysed,NaN
1,N002,A0897,distance_or_transport,0.95,Patient missed the appointment because they li...,Offer telehealth options for routine check-ups...,analysed,NaN
2,N003,A0526,NaN,NaN,NaN,NaN,invalid_response,"503 UNAVAILABLE. {'error': {'code': 503, 'mess..."
3,N004,A0926,NaN,NaN,NaN,NaN,skipped_blank,NaN
4,N005,A0534,illness_or_emergency,0.99,Patient missed the appointment due to a high f...,Contact the patient once they have recovered t...,analysed,NaN
5,N006,A0507,work_or_schedule,0.95,The patient missed the appointment due to an u...,Reschedule the appointment for an evening slot...,analysed,NaN
6,N007,A0333,forgot_or_no_reminder,0.95,Patient forgot the appointment because they di...,Reschedule the appointment and verify the pati...,analysed,NaN
7,N008,A0222,distance_or_transport,0.95,The patient missed the appointment due to a ci...,Reschedule the appointment and offer telehealt...,analysed,NaN
8,N009,A0977,other,0.50,Patient could not be reached via phone after t...,Attempt to contact the patient via alternative...,analysed,NaN
9,N010,A0596,NaN,NaN,NaN,NaN,invalid_response,"503 UNAVAILABLE. {'error': {'code': 503, 'mess..."


In [ ]:
results_df = pd.DataFrame(results)

display(results_df[results_df["status"] == "invalid_response"])

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,status,error
2,N003,A0526,NaN,NaN,NaN,NaN,invalid_response,"503 UNAVAILABLE. {'error': {'code': 503, 'mess..."
9,N010,A0596,NaN,NaN,NaN,NaN,invalid_response,"503 UNAVAILABLE. {'error': {'code': 503, 'mess..."


In [ ]:
for row in results:
    if row["status"] == "invalid_response":
        note_id = row["note_id"]
        note_text = notes_df.loc[
            notes_df["note_id"] == note_id, "note_text"
        ].iloc[0]

        try:
            raw_result = analyse_note(note_text)
            validated = NoteAnalysis.model_validate(raw_result)

            row.update(validated.model_dump())
            row["status"] = "analysed"
            row.pop("error", None)

        except Exception as e:
            row["error"] = str(e)

        time.sleep(5)

results_df = pd.DataFrame(results)
print(results_df["status"].value_counts())

status
analysed         9
skipped_blank    1
Name: count, dtype: int64


In [ ]:
results_df.to_csv("notes_analysed.csv", index=False)

print("Saved notes_analysed.csv")
print("Shape:", results_df.shape)

Saved notes_analysed.csv
Shape: (10, 7)


## 4.3 — Test Deliberately Invalid Replies and Compare Prompts

This section tests how the model handles invalid responses and compares prompt performance using notes N007 and N009.

In [ ]:
invalid_reply = {
    "root_cause": "forgot_or_no_reminder"
}

try:
    NoteAnalysis.model_validate(invalid_reply)
    print("Unexpected: invalid reply was accepted.")

except ValidationError as e:
    print("Validation correctly rejected the incomplete reply.")
    print(e)

Validation correctly rejected the incomplete reply.
3 validation errors for NoteAnalysis
confidence
  Field required [type=missing, input_value={'root_cause': 'forgot_or_no_reminder'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
summary
  Field required [type=missing, input_value={'root_cause': 'forgot_or_no_reminder'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
recommended_action
  Field required [type=missing, input_value={'root_cause': 'forgot_or_no_reminder'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


## 4.3 — Testing Prompt Weaknesses

In [ ]:
print("Shape:", results_df.shape)
print("Columns:", results_df.columns.tolist())

Shape: (10, 7)
Columns: ['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'status']


In [ ]:
print(results_df.columns.tolist())

['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'status']


In [ ]:
print(results[0].keys())

dict_keys(['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'status'])


In [ ]:
results_df = pd.DataFrame(results)

results_df["validation_status"] = results_df["status"]

def make_raw_response(row):
    if row["validation_status"] == "skipped_blank":
        return ""

    fields = [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]

    if all(pd.notna(row.get(field)) for field in fields):
        return json.dumps(
            {field: row[field] for field in fields},
            ensure_ascii=False
        )

    return ""

results_df["raw_response"] = results_df.apply(
    make_raw_response, axis=1
)

results_df = results_df.drop(columns=["status"])

results_df = results_df[
    [
        "note_id",
        "appointment_id",
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
        "validation_status",
        "raw_response"
    ]
]

print("Shape:", results_df.shape)
print("Columns:", results_df.columns.tolist())

Shape: (10, 8)
Columns: ['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'validation_status', 'raw_response']


In [ ]:
results_df.to_csv("notes_analysed.csv", index=False)

print("Saved successfully!")
print(results_df.shape)

Saved successfully!
(10, 8)


In [ ]:
display(
    notes_df[notes_df["note_id"].isin(["N007", "N009"])]
)

,note_id,appointment_id,note_text
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."


In [ ]:
note_007 = notes_df.loc[
    notes_df["note_id"] == "N007", "note_text"
].iloc[0]

note_009 = notes_df.loc[
    notes_df["note_id"] == "N009", "note_text"
].iloc[0]

print("N007:", note_007)
print("N009:", note_009)

N007: Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai.
N009: Phone was switched off, could not reach the patient after three attempts.


In [ ]:
invalid_reply_1 = {
    "root_cause": "transport_problem",
    "confidence": 1.5,
    "summary": "Patient missed the appointment.",
    "recommended_action": "Call the patient."
}

try:
    NoteAnalysis.model_validate(invalid_reply_1)
    print("Unexpected: invalid reply was accepted.")

except ValidationError as e:
    print("Test 1 passed: invalid reply rejected.")
    print(e)

Test 1 passed: invalid reply rejected.
2 validation errors for NoteAnalysis
root_cause
  Input should be 'forgot_or_no_reminder', 'distance_or_transport', 'cost_or_fee', 'illness_or_emergency', 'work_or_schedule' or 'other' [type=literal_error, input_value='transport_problem', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


In [ ]:
invalid_reply_2 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 0.8,
    "summary": "Patient forgot the appointment."
}

try:
    NoteAnalysis.model_validate(invalid_reply_2)
    print("Unexpected: invalid reply was accepted.")

except ValidationError as e:
    print("Test 2 passed: invalid reply rejected.")
    print(e)

Test 2 passed: invalid reply rejected.
1 validation error for NoteAnalysis
recommended_action
  Field required [type=missing, input_value={'root_cause': 'forgot_or...orgot the appointment.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [ ]:
def compare_prompt(note_text, prompt_style):

    if prompt_style == "basic":
        prompt = f"""
        Analyse this patient follow-up note and identify the reason
        for the missed appointment.

        Note: {note_text}

        Return JSON with root_cause, confidence, summary,
        and recommended_action.
        """

    else:
        prompt = f"""
        Context: A clinic wants to understand why patients miss appointments.
        Role: You are a patient follow-up note analyst.
        Action: Identify the most likely reason from the note.
        Note: {note_text}
        Tone: Clear and professional.
        Format: Return JSON with root_cause, confidence,
        summary (maximum 25 words), and recommended_action.
        Use only the allowed root_cause values:
        forgot_or_no_reminder, distance_or_transport, cost_or_fee,
        illness_or_emergency, work_or_schedule, other.
        Do not invent information.
        """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    return json.loads(response.text)


print("Prompt comparison function created.")

Prompt comparison function created.


In [ ]:
comparison_results = []

for note_id, note_text in [
    ("N007", note_007),
    ("N009", note_009)
]:
    for prompt_style in ["basic", "detailed"]:

        try:
            result = compare_prompt(note_text, prompt_style)
            validated = NoteAnalysis.model_validate(result)

            comparison_results.append({
                "note_id": note_id,
                "prompt_style": prompt_style,
                **validated.model_dump(),
                "status": "valid"
            })

            print(note_id, prompt_style, "— valid")

        except Exception as e:
            comparison_results.append({
                "note_id": note_id,
                "prompt_style": prompt_style,
                "status": "invalid",
                "error": str(e)
            })

        time.sleep(2)

comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)

N009 detailed — valid


,note_id,prompt_style,status,error,root_cause,confidence,summary,recommended_action
0,N007,basic,invalid,1 validation error for NoteAnalysis\nroot_caus...,NaN,NaN,NaN,NaN
1,N007,detailed,invalid,"503 UNAVAILABLE. {'error': {'code': 503, 'mess...",NaN,NaN,NaN,NaN
2,N009,basic,invalid,1 validation error for NoteAnalysis\nroot_caus...,NaN,NaN,NaN,NaN
3,N009,detailed,valid,NaN,other,0.9,Patient was unreachable because their phone wa...,Send an automated text message or mailer to re...


In [ ]:
for row in comparison_results:
    print(
        row["note_id"],
        row["prompt_style"],
        "—",
        row.get("error", "Valid")
    )
    print("-" * 50)

N007 basic — 1 validation error for NoteAnalysis
root_cause
  Input should be 'forgot_or_no_reminder', 'distance_or_transport', 'cost_or_fee', 'illness_or_emergency', 'work_or_schedule' or 'other' [type=literal_error, input_value='The patient forgot the a...receive a reminder SMS.', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error
--------------------------------------------------
N007 detailed — 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
--------------------------------------------------
N009 basic — 1 validation error for NoteAnalysis
root_cause
  Input should be 'forgot_or_no_reminder', 'distance_or_transport', 'cost_or_fee', 'illness_or_emergency', 'work_or_schedule' or 'other' [type=literal_error, input_value="The patient's phone was ...ct for the appointment.", input_type=str]
    

In [ ]:
for row in comparison_results:
    if row["status"] == "invalid" and "ValidationError" in row.get("error", ""):
        print(row["note_id"], row["prompt_style"])
        print(row["error"])
        print("-" * 50)

In [ ]:
for row in comparison_results:
    print(row["note_id"], row["prompt_style"], row["status"])
    if "error" in row:
        print(row["error"][:1000])
    print("-" * 40)

N007 basic invalid
1 validation error for NoteAnalysis
root_cause
  Input should be 'forgot_or_no_reminder', 'distance_or_transport', 'cost_or_fee', 'illness_or_emergency', 'work_or_schedule' or 'other' [type=literal_error, input_value='The patient forgot the a...receive a reminder SMS.', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error
----------------------------------------
N007 detailed invalid
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
----------------------------------------
N009 basic invalid
1 validation error for NoteAnalysis
root_cause
  Input should be 'forgot_or_no_reminder', 'distance_or_transport', 'cost_or_fee', 'illness_or_emergency', 'work_or_schedule' or 'other' [type=literal_error, input_value="The patient's phone was ...ct for the appointment.", input_type=str]
    Fo

In [ ]:
print("Prompt comparison summary")
print("Total comparison API calls:", len(comparison_results))
print("Valid responses:", sum(r["status"] == "valid" for r in comparison_results))
print("Invalid responses:", sum(r["status"] == "invalid" for r in comparison_results))
print("Pydantic invalid-reply tests passed: 2 out of 2")

Prompt comparison summary
Total comparison API calls: 4
Valid responses: 1
Invalid responses: 3
Pydantic invalid-reply tests passed: 2 out of 2


### Task 4.3 — Findings

Two deliberately incorrect replies were tested and correctly rejected by Pydantic validation.

The basic prompt produced two responses that failed validation. The detailed prompt produced one valid response and one 503 Service Unavailable error.

The detailed prompt provides clearer instructions, including allowed categories and output requirements. However, the 503 error is a temporary API service issue and not a prompt-quality failure.

A total of 4 API calls were made for the prompt comparison, with 1 valid response and 3 unsuccessful responses.

## 4.4 — Merge Analysis Results and Generate Recommendations

In [ ]:
import os

print(os.listdir())

['.config', 'appointments_clean.csv', 'follow up notes.csv', 'sample_data']


In [ ]:
print("notes_df available:", "notes_df" in globals())
print("results_df available:", "results_df" in globals())
print("results available:", "results" in globals())

notes_df available: False
results_df available: False
results available: False


In [ ]:
from typing import Literal

In [ ]:
# 1. Restore Gemini connection
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-flash-lite-latest"

# 2. Load original notes
notes_df = pd.read_csv("follow up notes.csv")
notes_to_process = notes_df.head(10).copy()

# 3. Define allowed output
SYSTEM_TEXT = """
You are a helpful assistant analysing patient appointment follow-up notes.
Identify the most likely reason for the missed appointment.
Use only these root_cause values:
forgot_or_no_reminder, distance_or_transport, cost_or_fee,
illness_or_emergency, work_or_schedule, other.
Return valid JSON matching the required schema.
Do not invent information.
"""

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause", "confidence", "summary",
        "recommended_action"
    ]
}

class NoteAnalysis(BaseModel):
    root_cause: Literal[
        "forgot_or_no_reminder",
        "distance_or_transport",
        "cost_or_fee",
        "illness_or_emergency",
        "work_or_schedule",
        "other"
    ]
    confidence: float = Field(ge=0, le=1)
    summary: str
    recommended_action: str

# 4. Analyse a note and preserve the original response
def analyse_note(note_text):
    prompt = f"""
    Context: A clinic wants to understand why patients miss appointments.
    Role: You are a patient follow-up note analyst.
    Action: Analyse the note and identify the most likely reason.
    Note: {note_text}
    Tone: Clear and professional.
    Format: Return JSON with root_cause, confidence, summary,
    and recommended_action. Keep summary within 25 words.
    Use only the allowed root_cause values.
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    return json.loads(response.text), response.text

# 5. Process first 10 notes
results = []
api_call_count = 0

for row in notes_to_process.itertuples(index=False):

    if pd.isna(row.note_text) or not str(row.note_text).strip():
        results.append({
            "note_id": row.note_id,
            "appointment_id": row.appointment_id,
            "root_cause": None,
            "confidence": None,
            "summary": None,
            "recommended_action": None,
            "validation_status": "skipped_blank",
            "raw_response": ""
        })
        continue

    success = False

    for attempt in range(3):
        try:
            api_call_count += 1
            raw_result, raw_text = analyse_note(row.note_text)
            validated = NoteAnalysis.model_validate(raw_result)

            results.append({
                "note_id": row.note_id,
                "appointment_id": row.appointment_id,
                **validated.model_dump(),
                "validation_status": "analysed",
                "raw_response": raw_text
            })

            success = True
            break

        except Exception as e:
            error_message = str(e)
            time.sleep(5)

    if not success:
        results.append({
            "note_id": row.note_id,
            "appointment_id": row.appointment_id,
            "root_cause": None,
            "confidence": None,
            "summary": None,
            "recommended_action": None,
            "validation_status": "invalid_response",
            "raw_response": error_message
        })

    time.sleep(1)

# 6. Save the recovered CSV
results_df = pd.DataFrame(results)

results_df = results_df[
    [
        "note_id",
        "appointment_id",
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
        "validation_status",
        "raw_response"
    ]
]

results_df.to_csv("notes_analysed.csv", index=False)

print("Gemini connection restored.")
print("API calls made:", api_call_count)
print("Records:", len(results_df))
print("CSV shape:", results_df.shape)
print(results_df["validation_status"].value_counts())
print("notes_analysed.csv saved successfully.")

Gemini connection restored.
API calls made: 11
Records: 10
CSV shape: (10, 8)
validation_status
analysed         9
skipped_blank    1
Name: count, dtype: int64
notes_analysed.csv saved successfully.


In [ ]:
appointments_df = pd.read_csv("appointments_clean.csv")

print("Appointments shape:", appointments_df.shape)
print("Columns:")
print(appointments_df.columns.tolist())

Appointments shape: (990, 16)
Columns:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [ ]:
print("Appointments columns:")
print(appointments_df.columns.tolist())

print("\nNotes columns:")
print(notes_df.columns.tolist())

Appointments columns:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']

Notes columns:
['note_id', 'appointment_id', 'note_text']


In [ ]:
print("appointment_id" in appointments_df.columns)
print("appointment_id" in results_df.columns)

True
True


In [ ]:
merged_df = appointments_df.merge(
    results_df,
    on="appointment_id",
    how="left"
)

print("Merged shape:", merged_df.shape)
print("Matched notes:", merged_df["note_id"].notna().sum())
print("Unmatched appointments:", merged_df["note_id"].isna().sum())

Merged shape: (990, 23)
Matched notes: 10
Unmatched appointments: 980


In [ ]:
print("Duplicate appointment IDs in appointments:",
      appointments_df["appointment_id"].duplicated().sum())

print("Duplicate appointment IDs in results:",
      results_df["appointment_id"].duplicated().sum())

Duplicate appointment IDs in appointments: 0
Duplicate appointment IDs in results: 0


In [ ]:
print("Appointments:", appointments_df.shape)
print("Analysis results:", results_df.shape)
print("Merged:", merged_df.shape)

print("Unique appointment IDs:",
      appointments_df["appointment_id"].nunique())

Appointments: (990, 16)
Analysis results: (10, 8)
Merged: (990, 23)
Unique appointment IDs: 990


In [ ]:
display(
    merged_df[
        [
            "appointment_id",
            "no_show",
            "note_id",
            "root_cause",
            "confidence",
            "recommended_action"
        ]
    ].dropna(subset=["note_id"])
)

,appointment_id,no_show,note_id,root_cause,confidence,recommended_action
218,A0222,1,N008,distance_or_transport,0.95,Reschedule the appointment and offer telehealt...
328,A0333,1,N007,forgot_or_no_reminder,0.95,Reschedule the appointment and verify the pati...
500,A0507,1,N006,work_or_schedule,0.95,Reschedule the appointment for an evening slot...
518,A0526,1,N003,cost_or_fee,0.95,Provide information on available financial ass...
526,A0534,1,N005,illness_or_emergency,1.00,Reschedule the appointment after the patient h...
587,A0596,1,N010,work_or_schedule,0.95,Reschedule the appointment for a Saturday as r...
861,A0872,1,N001,forgot_or_no_reminder,0.99,Reschedule the appointment for next week and v...
886,A0897,1,N002,distance_or_transport,0.95,Offer telehealth options or assistance with tr...
915,A0926,1,N004,None,NaN,None
966,A0977,1,N009,other,0.50,Send a follow-up letter or message by alternat...


In [ ]:
root_cause_summary = (
    merged_df["root_cause"]
    .dropna()
    .value_counts()
    .reset_index()
)

root_cause_summary.columns = [
    "root_cause",
    "number_of_notes"
]

display(root_cause_summary)

,root_cause,number_of_notes
0,distance_or_transport,2
1,forgot_or_no_reminder,2
2,work_or_schedule,2
3,cost_or_fee,1
4,illness_or_emergency,1
5,other,1


In [ ]:
print(root_cause_summary.to_string(index=False))

           root_cause  number_of_notes
distance_or_transport                2
forgot_or_no_reminder                2
     work_or_schedule                2
          cost_or_fee                1
 illness_or_emergency                1
                other                1


In [ ]:
recommendations = {
    "distance_or_transport":
        "Explore transport support, remote consultations, or flexible clinic locations.",

    "forgot_or_no_reminder":
        "Send SMS reminders before appointments and confirm patient contact details.",

    "work_or_schedule":
        "Offer flexible appointment slots, including suitable after-work timings.",

    "cost_or_fee":
        "Communicate consultation fees clearly and explore affordable payment options.",

    "illness_or_emergency":
        "Provide an easy rescheduling process for patients facing illness or emergencies.",

    "other":
        "Contact the patient to understand the reason and offer appropriate support."
}

root_cause_summary["recommended_action"] = (
    root_cause_summary["root_cause"].map(recommendations)
)

display(root_cause_summary)

,root_cause,number_of_notes,recommended_action
0,distance_or_transport,2,"Explore transport support, remote consultation..."
1,forgot_or_no_reminder,2,Send SMS reminders before appointments and con...
2,work_or_schedule,2,"Offer flexible appointment slots, including su..."
3,cost_or_fee,1,Communicate consultation fees clearly and expl...
4,illness_or_emergency,1,Provide an easy rescheduling process for patie...
5,other,1,Contact the patient to understand the reason a...


In [ ]:
merged_df.to_csv("appointments_with_ai_analysis.csv", index=False)

root_cause_summary.to_csv(
    "clinic_recommendations.csv",
    index=False
)

print("Both files saved successfully.")

Both files saved successfully.


In [ ]:
print("Appointments with AI analysis:",
      os.path.exists("appointments_with_ai_analysis.csv"))

print("Clinic recommendations:",
      os.path.exists("clinic_recommendations.csv"))

Appointments with AI analysis: True
Clinic recommendations: True


In [ ]:
from google.colab import files

files.download("appointments_with_ai_analysis.csv")
files.download("clinic_recommendations.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemi